# Laporan Praktikum Association Rule Mining — Tugas 1 sampai 4
**Algoritma:** Apriori dan FP-Growth | **Lingkungan:** Google Colab / Jupyter Notebook

**PENTING — Asal dataset:** Modul dosen menyebut 60 transaksi toko kelontong, tetapi hanya memuat lima transaksi pertama. File `Dataset_Tugas_1_2_3_Kelontong_Sintetis.csv` adalah **dataset pengganti sintetis**, bukan dataset asli modul; lima transaksi pertama dibuat sama dengan contoh modul. Oleh karena itu, hasil perhitungan tugas 1–3 **tidak boleh diklaim sama** dengan keluaran pada Bagian 4 modul. Angka acuan modul dicantumkan terpisah sebagai pembanding metodologis. Tugas 4 menggunakan dataset Pet Shop sintetis 60 transaksi dari pekerjaan sebelumnya.

**Cara menjalankan:** Upload kedua file CSV ke sesi Colab, lalu jalankan semua sel dari atas ke bawah. Seluruh tabel dan grafik dihitung ulang dari file yang diunggah.

## Persiapan: pustaka dan pemuatan data

In [ ]:
%pip -q install mlxtend pandas numpy matplotlib
import pandas as pd, numpy as np, matplotlib.pyplot as plt, time
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from IPython.display import display

def load_transactions(path):
    raw = pd.read_csv(path, encoding='utf-8-sig')
    assert {'ID_Transaksi','Items'}.issubset(raw.columns)
    tx = [[v.strip() for v in s.split(',') if v.strip()] for s in raw['Items'].fillna('')]
    assert all(tx), 'Ada transaksi kosong'
    te = TransactionEncoder()
    matrix = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
    return raw, tx, matrix

def make_rules(fi, threshold=0.5):
    if fi.empty or not any(len(x)>1 for x in fi.itemsets):
        return pd.DataFrame()
    return association_rules(fi, metric='confidence', min_threshold=threshold).sort_values('lift',ascending=False).reset_index(drop=True)

def pretty_rules(r):
    if r.empty: return pd.DataFrame()
    x = r[['antecedents','consequents','support','confidence','lift','leverage','conviction']].copy()
    for col in ['antecedents','consequents']:
        x[col] = x[col].map(lambda s:', '.join(sorted(s)))
    return x.round(4)

raw, transactions, df = load_transactions('Dataset_Tugas_1_2_3_Kelontong_Sintetis.csv')
print('Dataset kelontong:', len(transactions),'transaksi,',df.shape[1],'item')
display(raw.head(), df.head())
print('Frekuensi item:')
display(df.sum().sort_values(ascending=False).rename('Jumlah transaksi').to_frame())

## Tugas 1 — Eksplorasi ambang batas
**1.1** Jalankan `min_support=0.10`, `min_confidence=0.50`, lalu bandingkan dengan `min_support=0.15` pada dataset yang sama. **Acuan modul (dataset asli yang tidak tersedia):** 0.10 menghasilkan 19 itemset/20 aturan; 0.15 menghasilkan 17 itemset/18 aturan. Angka tersebut bukan hasil dataset pengganti.

In [ ]:
fi10 = apriori(df,min_support=0.10,use_colnames=True)
fi15 = apriori(df,min_support=0.15,use_colnames=True)
r10 = make_rules(fi10,0.5)
r15 = make_rules(fi15,0.5)
comparison = pd.DataFrame({'min_support':[0.10,0.15], 'Jumlah itemset':[len(fi10),len(fi15)],'Jumlah aturan':[len(r10),len(r15)]})
display(comparison)
print('Perubahan itemset:',len(fi10)-len(fi15),'| Perubahan aturan:',len(r10)-len(r15))

**1.2** Naikkan `min_confidence` dari 0.50 menjadi 0.70 pada `min_support=0.15`. Tampilkan aturan yang lolos dan gugur (hasil dari dataset pengganti).

In [ ]:
r70 = make_rules(fi15,0.7)
failed = r15[r15['confidence'] < 0.7]
print('Aturan awal:',len(r15),'| Lolos:',len(r70),'| Gugur:',len(failed))
print('Aturan yang gugur:')
display(pretty_rules(failed))
print('Aturan yang lolos:')
display(pretty_rules(r70))

**1.3** Grafik sweep `min_confidence` pada `min_support=0.15`.

In [ ]:
thresholds=[0.5,0.6,0.7,0.8,0.9,1.0]
counts=[len(make_rules(fi15,c)) for c in thresholds]
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(thresholds,counts,marker='o');ax.set(xlabel='min_confidence',ylabel='Jumlah aturan',title='Tugas 1: Pengaruh min_confidence');ax.grid(alpha=.3)
for x,y in zip(thresholds,counts):ax.annotate(str(y),(x,y),xytext=(0,7),textcoords='offset points',ha='center')
plt.tight_layout();plt.show()
display(pd.DataFrame({'min_confidence':thresholds,'jumlah_aturan':counts}))

**1.4 Analisis trade-off:** Ambang `min_support` dan `min_confidence` rendah dapat mengungkap lebih banyak kombinasi, tetapi beberapa aturan mungkin didukung sedikit transaksi atau kurang konsisten. Ambang tinggi menyaring aturan yang lemah, namun berisiko menghilangkan pola yang relevan pada produk jarang dibeli. Perhatikan jumlah aturan dari grafik di atas sebelum menetapkan ambang. Confidence tinggi saja tidak cukup; periksa pula support dan lift. Ini adalah asosiasi, bukan bukti kausalitas.

## Tugas 2 — Leverage dan Conviction
Pilih dua aturan yang **tidak dibahas pada Bagian 4.6 modul**: `Mie Instan → Telur` dan `Roti → Susu`. Pada dataset sintetis, keduanya dihitung dari transaksi aktual dan ditampilkan meskipun mungkin tidak lolos batas tertentu.

Rumus: **Leverage** = support(X∪Y) − support(X)×support(Y). **Conviction** = (1−support(Y)) / (1−confidence(X→Y)); jika confidence = 1, conviction = ∞.

In [ ]:
def direct_metrics(tx, X, Y):
    n=len(tx); X=set(X);Y=set(Y)
    sx=sum(X.issubset(set(t)) for t in tx)/n
    sy=sum(Y.issubset(set(t)) for t in tx)/n
    sxy=sum((X|Y).issubset(set(t)) for t in tx)/n
    conf=sxy/sx if sx else float('nan')
    lift=conf/sy if sy else float('nan')
    lev=sxy-sx*sy
    conv=(1-sy)/(1-conf) if conf<1 else float('inf')
    return {'Aturan':f"{', '.join(X)} → {', '.join(Y)}",'support':sxy,'confidence':conf,'lift':lift,'leverage':lev,'conviction':conv}
selected=pd.DataFrame([direct_metrics(transactions,['Mie Instan'],['Telur']),direct_metrics(transactions,['Roti'],['Susu'])])
display(selected.round(4))

**Interpretasi bisnis:** (1) Untuk `Mie Instan → Telur`, pertimbangkan penempatan telur dekat rak mi instan atau kupon tambahan telur ketika pembeli memasukkan mi instan ke keranjang. (2) Untuk `Roti → Susu`, uji promosi sarapan berupa diskon susu bagi pembeli roti. **Leverage positif** berarti kemunculan bersama lebih sering daripada perkiraan independen; **Lift > 1** menunjukkan asosiasi positif; **Conviction > 1** mengindikasikan aturan lebih baik daripada prediksi berbasis prevalensi consequent. Evaluasi nilai yang benar-benar tercetak, bukan hanya asumsi ini.

## Tugas 3 — Apriori vs FP-Growth pada `min_support=0.05`

In [ ]:
def benchmark(matrix,support,repeats=15):
    records=[]
    for method,func in [('Apriori',apriori),('FP-Growth',fpgrowth)]:
        durations=[];result=None
        for _ in range(repeats):
            t0=time.perf_counter();result=func(matrix,min_support=support,use_colnames=True)
            durations.append((time.perf_counter()-t0)*1000)
        records.append({'min_support':support,'Algoritma':method,'Jumlah itemset':len(result),'Median waktu (ms)':np.median(durations),'Itemsets':set(map(frozenset,result.itemsets))})
    assert records[0]['Itemsets']==records[1]['Itemsets'],'Hasil kedua algoritma tidak identik!'
    return pd.DataFrame([{k:v for k,v in r.items() if k!='Itemsets'} for r in records])
bench=pd.concat([benchmark(df,0.05),benchmark(df,0.15)],ignore_index=True)
display(bench.round(4))
print('Acuan waktu dari modul untuk dataset asli pada support 0.15: Apriori 10.42 ms, FP-Growth 2.95 ms; tidak dapat dibandingkan langsung karena dataset/perangkat berbeda.')
for s in [0.05,0.15]:
    part=bench[bench.min_support==s].set_index('Algoritma')
    a=part.loc['Apriori','Median waktu (ms)'];f=part.loc['FP-Growth','Median waktu (ms)']
    print(f'support={s:.2f}: FP-Growth {a/f:.2f}× lebih cepat daripada Apriori' if f<a else f'support={s:.2f}: Apriori {f/a:.2f}× lebih cepat daripada FP-Growth')
fig,ax=plt.subplots(figsize=(8,4))
for name,g in bench.groupby('Algoritma'):
    ax.plot(g.min_support,g['Median waktu (ms)'],marker='o',label=name)
ax.set(xlabel='min_support',ylabel='Median waktu eksekusi (ms)',title='Tugas 3: Perbandingan runtime');ax.legend();ax.grid(alpha=.3);plt.tight_layout();plt.show()

**Analisis:** Pada ambang 0.05 lebih banyak kombinasi berpotensi memenuhi syarat dibanding 0.15, sehingga Apriori mungkin menguji lebih banyak kandidat. FP-Growth mengompresi transaksi dalam FP-tree sehingga sering lebih efisien pada data besar atau padat. Namun **dataset hanya 60 transaksi**: overhead pustaka dan variasi lingkungan dapat membuat perbedaan runtime kecil atau bahkan terbalik. Gunakan tabel median dari 15 pengulangan di atas untuk menjawab apakah keunggulan FP-Growth makin terlihat pada dataset ini; jangan menganggap hasilnya pasti.

## Tugas 4 — Studi kasus mandiri: Pet Shop
Dataset sintetis 60 transaksi, 10 jenis produk. Parameter dipilih `min_support=0.15` (setidaknya 9 dari 60 transaksi) dan `min_confidence=0.50` (sedikitnya separuh transaksi antecedent mengandung consequent).

In [ ]:
pet_raw,pet_tx,pet_df=load_transactions('Dataset_Tugas_4_Pet_Shop.csv')
print('Jumlah transaksi:',len(pet_tx),'| Jumlah jenis produk:',pet_df.shape[1]);display(pet_raw.head());display(pet_df.head())
pet_fi=apriori(pet_df,min_support=0.15,use_colnames=True).sort_values('support',ascending=False)
pet_rules=make_rules(pet_fi,0.50)
print('Frequent itemset:',len(pet_fi),'| Aturan:',len(pet_rules))
display(pet_fi.head(15),pretty_rules(pet_rules).head(15))

**Visualisasi 1:** Scatter plot Support vs Confidence, warna dan ukuran gelembung merepresentasikan Lift.

In [ ]:
if len(pet_rules):
    fig,ax=plt.subplots(figsize=(8,5))
    dots=ax.scatter(pet_rules.support,pet_rules.confidence,c=pet_rules.lift,s=pet_rules.lift*90,cmap='viridis',alpha=.8,edgecolor='black',linewidth=.3)
    fig.colorbar(dots,ax=ax,label='Lift');ax.set(xlabel='Support',ylabel='Confidence',title='Pet Shop — Support vs Confidence');ax.grid(alpha=.2)
    plt.tight_layout();plt.show()

**Visualisasi 2:** Delapan aturan dengan Lift tertinggi.

In [ ]:
if len(pet_rules):
    top=pretty_rules(pet_rules).head(8).copy()
    top['Aturan']=top['antecedents']+' → '+top['consequents']
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(top['Aturan'][::-1],top['lift'][::-1]);ax.set(xlabel='Lift',title='Pet Shop — 8 Aturan Lift Tertinggi')
    plt.tight_layout();plt.show()
    display(top[['Aturan','support','confidence','lift']])

**Rekomendasi bisnis (minimal 3):**
1. **Bundling perlengkapan mandi:** Jika aturan `Shampo Hewan + Mainan Hewan → Sisir Hewan` muncul dengan Lift tinggi, buat paket grooming berisi ketiganya dan uji diskon kecil.
2. **Cross-selling kebutuhan kucing:** Jika aturan `Makanan Kucing → Pasir Kucing` memiliki Lift > 1, tampilkan rekomendasi pasir kucing di kasir/checkout untuk pembeli makanan kucing.
3. **Penataan rak kebutuhan anjing:** Jika aturan `Makanan Anjing → Snack Anjing` memiliki Lift > 1, tempatkan snack anjing berdekatan dengan makanan anjing.

**Validasi:** Rekomendasi di atas merupakan hipotesis berbasis kelompok produk; **pastikan** aturan tersebut benar-benar ada dan memiliki Lift > 1 pada tabel output. Jika tidak, gunakan tiga aturan dengan Lift tertinggi yang benar-benar muncul di bawah ini.

In [ ]:
if len(pet_rules):
    print('Tiga aturan Lift tertinggi yang TERBUKTI pada dataset:')
    display(pretty_rules(pet_rules).head(3))
    pretty_rules(pet_rules).to_csv('Hasil_Aturan_Pet_Shop.csv',index=False)
    print('Disimpan: Hasil_Aturan_Pet_Shop.csv')

## Kesimpulan
- Association rule mining menemukan pola pembelian bersama dari transaksi, bukan sebab-akibat.
- Pengaturan support dan confidence mengubah banyaknya pola yang ditemukan; Lift membantu mengukur kekuatan hubungan relatif terhadap frekuensi consequent.
- Apriori dan FP-Growth harus menghasilkan himpunan frequent itemset yang sama pada ambang yang sama, walaupun waktu eksekusinya dapat berbeda.
- Dataset Pet Shop dapat digunakan untuk menyusun strategi bundling, cross-selling, dan penataan rak yang perlu diuji pada data transaksi nyata.

**Referensi:** Modul Praktikum Data Mining — Association Rule Mining (Universitas Negeri Padang, 2026); Agrawal & Srikant (1994); Han, Pei, & Yin (2000); Raschka (2018).